# SSAG Motion Studio — free GPU notebook (experimental)

**Real dance generation, not a simulation.** Uses open-source [Motion Mirror](https://github.com/halli75/motion-mirror) with **Wan2.1 VACE** to transfer motion from an authorized 5-second reference video to a photograph.

1. **Make this notebook PRIVATE** before using a child's or client's photo; no public datasets, notebook sharing or publishing source outputs.
2. Open in Kaggle. In Settings select a free **GPU T4 x2** if available, enable Internet (needed to install libraries/download model), and start a live session.
3. Run the resource check first. This model expects approximately 9 GB GPU VRAM, 32 GB system RAM and 20 GB disk cache. Kaggle's free machines may not meet every requirement and can be unavailable; no success guarantee.
4. Choose and locally upload your **own** photograph and a dance clip you have the right to use. Do not upload copyrighted music/video without appropriate permission.
5. Run the last cell. It only declares success after finding a nonempty generated MP4.

**Not a production backend.** Kaggle free GPU sessions and quotas are not an always-on hosted service. For a business storefront, use a properly provisioned private GPU worker. No external uploads are performed automatically by this notebook except necessary downloads of public AI model weights from their publishers.

**Avoid:** 1.3B `--fast` mode, whose optional distilled weights have noncommercial terms. The normal Wan2.1 VACE weights are offered under Apache 2.0; review downstream licenses for each use.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json
from IPython.display import display
import torch
try:
    import psutil
except ImportError:
    psutil = None
gpu_ok=torch.cuda.is_available()
gpu_mem=(torch.cuda.get_device_properties(0).total_memory/1024**3 if gpu_ok else 0)
ram_gb=(psutil.virtual_memory().total/1024**3 if psutil else 0)
disk_gb=shutil.disk_usage("/kaggle/working").free/1024**3
print(json.dumps({"gpu":torch.cuda.get_device_name(0) if gpu_ok else "NO CUDA GPU",
                  "gpu_vram_gib":round(gpu_mem,1),"host_ram_gib":round(ram_gb,1),
                  "free_disk_gib":round(disk_gb,1)},indent=2))
assert gpu_ok and gpu_mem >= 9, "No usable GPU/VRAM. Stop; choose GPU T4 if permitted."
if ram_gb < 31:
    print("WARNING: upstream recommends ~32 GB system RAM. Free Kaggle hosts may have less and real generation may fail.")
assert disk_gb >= 23, "Not enough free disk to install model and store the output."


In [ ]:
# INSTALL — may take a while and is subject to notebook internet/GPU quotas.
import subprocess, sys, os
root="/kaggle/working/motion-mirror"
if not os.path.isdir(root):
    subprocess.run(["git","clone","--depth","1","https://github.com/halli75/motion-mirror.git",root],check=True)
subprocess.run([sys.executable,"-m","pip","install","-e",root+"[cuda,gpu-inference]"],check=True)
cmd=["motion-mirror","download","--model"]
for model in ("dwpose","wan-1.3b-vace"):
    subprocess.run(cmd+[model],check=True)
print("Software and models downloaded (generation still unverified).")


In [ ]:
# PRIVATE SESSION UPLOAD — do not save images in a public Kaggle dataset.
# Run this cell, choose one full-body photo and one reference dance video.
from ipywidgets import FileUpload, VBox, HTML
photo_upload=FileUpload(accept=".jpg,.jpeg,.png,.webp",multiple=False,description="Person photo")
dance_upload=FileUpload(accept=".mp4,.webm,.mov",multiple=False,description="Dance video")
display(VBox([HTML("<b>Choose person photo (guardian/subject consent required)</b>"),photo_upload,
              HTML("<b>Choose permitted dance-reference video</b>"),dance_upload]))


In [ ]:
from pathlib import Path
from PIL import Image
from io import BytesIO

def first_upload(widget):
    v=widget.value
    if not v:
        raise ValueError("File was not selected.")
    if isinstance(v,dict): # widget v7 format
        name,details=next(iter(v.items()))
        return name,bytes(details["content"])
    details=v[0] # widget v8 format
    return details["name"],bytes(details["content"])

name,photo_bytes=first_upload(photo_upload)
dance_name,dance_bytes=first_upload(dance_upload)
assert len(photo_bytes)<=15*1024*1024, "Photo exceeds 15 MB"
assert len(dance_bytes)<=80*1024*1024, "Reference exceeds 80 MB"
assert Path(dance_name).suffix.lower() in (".mp4",".webm",".mov"), "Not a supported reference video"
input_dir=Path("/kaggle/working/ssag-private-input")
input_dir.mkdir(parents=True,exist_ok=True)
with Image.open(BytesIO(photo_bytes)) as im:
    im.convert("RGB").save(input_dir/"character.png")
ref=input_dir/"motion-reference"+Path(dance_name).suffix.lower()
ref.write_bytes(dance_bytes)
print("Photo and reference saved to this PRIVATE temporary GPU session, not to GitHub.")


In [ ]:
from IPython.display import Video, display, FileLink
from pathlib import Path
import subprocess, shutil
output_dir=Path("/kaggle/working/ssag-rendered")
output_dir.mkdir(exist_ok=True)
cmd=["motion-mirror","run",str(input_dir/"character.png"),str(ref),
     "--backend","wan-1.3b-vace","--offload-model","--t5-cpu",
     "--frames","81","--resolution","480x832","--output-dir",str(output_dir)]
print("Starting REAL GPU motion transfer; no mock/fake renderer.")
result=subprocess.run(cmd,check=False)
if result.returncode != 0:
    raise RuntimeError(f"Real renderer failed (code {result.returncode}); no generated video.")
videos=sorted(output_dir.rglob("*.mp4"),key=lambda f:f.stat().st_mtime,reverse=True)
if not videos or videos[0].stat().st_size < 1024:
    raise RuntimeError("No real output MP4 detected.")
video=videos[0]
print("Actual rendered MP4:",video,"size:",video.stat().st_size,"bytes")
display(Video(filename=str(video),embed=True))
display(FileLink(str(video),result_html_prefix="Download the rendered MP4: "))


### Cleanup and reporting

After downloading your actual rendered clip, delete the private input/output files or shut down the Kaggle session. Do not share the notebook output publicly. A real sample must be examined for face fidelity, clothing and motion quality before using it for a customer.

Report to the SSAG owner dashboard only after an actual MP4 passes quality review; a successful install does **not** mean customer video generation is ready.